# The limit order book

Section 1.1 of the lecture notes, at the keyboard.  The definitions are stated there; here we
build the object they describe, read the derived quantities off it, and watch one order at a
time change it.

Prices are integer counts of ticks throughout.  The conversion to currency happens once, at
the boundary, and nowhere else.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from unito26.lob import frames, lobster
from unito26.lob.lobster_session import LobsterMarketSession
from unito26.lob.messages import (
    BUY, SELL, GridDepth, ReportedDepth, SweepSize,
    TickGrid, limit_order, market_order, withdrawal,
)
from unito26.lob.orderbook import AggregateBook
from unito26.lob.statistics import SessionStatistics
from unito26.lob.visualization import (
    ascii_ladder, book_figure, describe_message, example_figure, imbalance_figure,
    level_evolution_figure, snapshots_figure, touch_figure, use_template,
)
from unito26.lob.worked_examples import CATALOGUE, BASELINE_BOOK, check, to_sides

use_template()

GRID = TickGrid(0.01)
DEPTH = ReportedDepth(10)


## 1. An order is four numbers

A limit order is the tuple $(t, q, p, d)$: time, size, price, direction.  The direction is an
`int`, $+1$ to buy and $-1$ to sell, because the single expression $pd$ collapses both sides
of the book into one comparison.

In [ ]:
for message in (
    limit_order(0.0, 250, 999, SELL),
    market_order(0.0, 250, SELL),
    market_order(0.0, 250, BUY),
    withdrawal(0.0, 60, 999, BUY),
):
    print(f"{describe_message(message):<22} {message.kind.name:<8} price={message.price}")

A market order is not a third kind of message.  It is a limit order carrying a price
*specification* that guarantees execution: $0$ for a sell, and the largest representable
integer for a buy.  Neither is a price at which anything may rest — which is why the buy
prints as `9223372036854775807` and why `describe_message` does not show it.

The tick grid is the boundary.  It converts currency to ticks on the way in, and it rejects a
price that is not a multiple of the tick rather than rounding it: a rounded price is a level
that fails to compare equal to the one it should have been.

In [ ]:
print("10.02 ->", GRID.to_ticks(10.02), "ticks")
print("1002  ->", GRID.to_price(1002))

try:
    GRID.to_ticks(10.025)
except ValueError as error:
    print("off the grid:", error)

## 2. One order meets a book

The book of §1.1 is the aggregated one: a map from price to the total size resting there, and
nothing about which orders make up that total.  We use the worked example of the notes.

In [ ]:
book = AggregateBook.from_levels(*to_sides(BASELINE_BOOK))
print(ascii_ladder(book))

Now one sell order for 400 shares, priced at 999.  It is priced into the bid side, so it
trades before it rests.

In [ ]:
result = book.submit(limit_order(1.0, 400, 999, SELL), record=True)

for fill in result.fills:
    print(f"filled {fill.size:>4} at {fill.price}")
print(f"q_M = {result.market_order_size}, rested = {400 - result.market_order_size}")
print()
print(ascii_ladder(book))

The bid side lost two whole levels, so the best bid moved from 1000 down to 998.  The 100
shares that could not be filled came to rest at 999 — on the **ask** side, a tick below the
old best bid, on a level this order had just cleared.  Every ask price therefore shifted:
1002 was the best ask and is now three ticks away from it.

Each fill printed at the **resting** order's price, never at the 999 the incoming order named.

In [ ]:
book_figure(book, depth=6, title="after the sell of 400 at 999")

## 3. The grid is not a list of queues

Level $i$ is a *position on the price grid*, occupied or not.  A market data feed reports the
first $L$ prices that *carry size*.  The two agree only on a book with no holes, and the book
we just made has two.

In [ ]:
print("grid positions  :", book.levels(SELL, 6))
print("occupied levels :", book.occupied_levels(SELL, ReportedDepth(6)))

Three of the six grid positions read zero, and they are not the same kind of zero.  The zeros
at 1000 and 1001 sit *between* two occupied prices; the one at 1004 sits past the last of
them.  Only the first kind is a gap, which is why `empty_grid_positions` returns two indices
and not three.

In [ ]:
print("empty grid positions:", book.empty_grid_positions(SELL, ReportedDepth(6)))
print("gaps                :", book.gap_count(SELL, ReportedDepth(6)))
print("occupied levels     :", book.occupied_level_count(SELL, ReportedDepth(6)))
print("grid span they cover:", book.grid_span(SELL, ReportedDepth(6)))

Three occupied levels spanning five grid positions.  Any quantity indexed by level means
something different under the two readings, and §1.2 is where that difference is paid for.

## 4. What one configuration yields

Spread, mid-price, micro-price, queue imbalance and sweep cost are all functions of the state
the book already holds.  None of them needs the tape.

In [ ]:
fresh = AggregateBook.from_levels(*to_sides(BASELINE_BOOK))

print("best bid / best ask:", fresh.best_bid_price, "/", fresh.best_ask_price)
print("spread             :", fresh.spread)
print("mid-price          :", fresh.mid_price)
print("micro-price        :", round(fresh.micro_price, 4))
for n in (1, 2, 3):
    print(f"queue imbalance I^{n}:", round(fresh.queue_imbalance(GridDepth(n)), 4))

The imbalance is bid minus ask over the total, so a bid-heavy book is positive.  Here the ask
is heavier at the touch and $I^1$ is negative.

The micro-price has a second expression: the weighting in which each price carries the size
resting on the *opposite* side.  Computing it that way is an independent route to the same
number.

In [ ]:
bid_size, ask_size = fresh.best_bid_size, fresh.best_ask_size
opposite_weighted = (bid_size * fresh.best_ask_price + ask_size * fresh.best_bid_price) / (
    bid_size + ask_size
)
print("by definition:", fresh.micro_price)
print("by weighting :", opposite_weighted)
assert abs(fresh.micro_price - opposite_weighted) < 1e-12

### Walking the book

The sweep cost is the per-share cost of a market order, in ticks, measured against the mid.
Every share pays the half-spread; each additionally pays its own distance from the touch.

The book knows whether an order walked — `walked_the_book` asks whether the fills printed at
more than one price — so the criterion and the cost are two independent readings, and we can
put them side by side.

In [ ]:
half_spread = fresh.spread / 2
print(f"half-spread: {half_spread}\n")

for size in (50, 100, 120, 200, 300):
    probe = AggregateBook.from_levels(*to_sides(BASELINE_BOOK))
    filled = probe.submit(market_order(1.0, size, BUY), record=True)
    cost = fresh.sweep_cost(BUY, SweepSize(size), ReportedDepth(10))
    print(f"buy {size:>3}: sweep cost {cost:.4f}   walked: {filled.walked_the_book}")

The two columns agree, and they agree at the boundary: 120 clears the touch exactly, prints at
one price, and costs exactly the half-spread.  An order walks the book precisely when its
sweep cost exceeds the half-spread — the walk term being positive exactly when some share is
taken at a level past the touch.

The equivalence holds only where the sweep cost is defined, and it is not defined for an order
larger than the side.

In [ ]:
ask_side = sum(size for _, size in fresh.occupied_levels(SELL, ReportedDepth(10)))
print("ask side holds   :", ask_side)
print("sweep cost of 400:", fresh.sweep_cost(BUY, SweepSize(400), ReportedDepth(10)))

`None`, and not a number.  The quantity is defined only where the side holds at least the
shares asked for, and code must branch on that rather than return something plausible.

## 5. Every limit order is a market order and a resting order

Processing $(t,q,p,-1)$ is processing the pair $[(t,q_M,0,-1), (t,q-q_M,p,-1)]$: the market
part, then the resting part.  The book reports $q_M$ directly, so we can check it against the
definition read off the opposite side.

In [ ]:
def price_eligible_size(book, price, direction):
    '''Resting size on the far side that an order at ``price`` may take.'''
    opposite = book.levels_map(-direction)
    return sum(
        resting
        for level, resting in opposite.items()
        if level * direction <= price * direction
    )


probe = AggregateBook.from_levels(*to_sides(BASELINE_BOOK))
predicted = min(400, price_eligible_size(probe, price=999, direction=SELL))
executed = probe.submit(limit_order(1.0, 400, 999, SELL), record=True).market_order_size

print("q_M by definition:", predicted)
print("q_M by the engine:", executed)

A matching engine therefore needs **one** code path.  The loop that consumes the opposite side
simply does not execute when nothing matches, so a passive order takes the same route as an
aggressive one.

In [ ]:
passive = AggregateBook.from_levels(*to_sides(BASELINE_BOOK))
joined = passive.submit(limit_order(2.0, 75, 999, BUY), record=True)

print("fills       :", joined.fills)
print("q_M         :", joined.market_order_size)
print("bid side now:", passive.levels_map(BUY))

No fills, one code path, and the 75 shares joined the queue already resting at 999.

## 6. The catalogue

Eleven transitions, one for each branch of the update rule.  Having one example per branch is
what makes it possible to argue the set is complete rather than merely plausible.  `check`
asserts the resulting state, which is derived from the notation rather than captured from a
run.

In [ ]:
for example in CATALOGUE:
    check(AggregateBook, example)
    print(f"  {example.name}")
print(f"\n{len(CATALOGUE)} examples, all passing")

In [ ]:
example_figure(next(e for e in CATALOGUE if "walks the book" in e.name), depth=7)

## 7. Two things the aggregated state cannot answer

**Whose size it is.**  A sum does not record its summands.  Fold two different streams of
orders — one submission of 100, or five of 20 — and the book that results is the same book.

In [ ]:
one_order = [limit_order(1.0, 100, 1000, BUY)]
five_orders = [limit_order(1.0 + i, 20, 1000, BUY) for i in range(5)]

left, right = AggregateBook(), AggregateBook()
for stream, target in ((one_order, left), (five_orders, right)):
    for message in stream:
        target.apply(message, record=False)

print("one submission of 100:", left.levels_map(BUY))
print("five submissions of 20:", right.levels_map(BUY))
print("same state:", left.levels_map(BUY) == right.levels_map(BUY))

So no question that names an order can be answered from this state: how much size is ahead of
mine, will my order be filled, whose fill was that, what is one account's position.  Those
need the order-level book, which carries identity and which we do not build here.

**How much traded.**  Two books can reach the same state by different routes, one of which
traded and one of which did not.

In [ ]:
executed, cancelled = (AggregateBook.from_levels(*to_sides(BASELINE_BOOK)) for _ in range(2))

executed.apply(market_order(1.0, 40, SELL), record=False)
cancelled.apply(withdrawal(1.0, 40, 1000, BUY), record=False)

print("after a market sell of 40:", executed.levels_map(BUY), "traded", executed.last_traded_size)
print("after a withdrawal of 40 :", cancelled.levels_map(BUY), "traded", cancelled.last_traded_size)
print("same state:", executed.levels_map(BUY) == cancelled.levels_map(BUY))

Same state, and 40 shares traded against none.  A level shrinks by cancellation as well as by
execution, so **no sequence of sizes determines the volume traded** — which is why volume is
read off the tape, as $\sum q_M$ over the market orders of a window, and never differenced out
of the book.

What survives is exactly what this chapter is for: how size has accumulated across the levels
of the two sides.


## 8. A real book

Every book so far is one we wrote down.  What follows is the same objects, read off a recorded
day.

The data is [LOBSTER](https://lobsterdata.com)'s free sample: one day on NASDAQ, 21 June 2012,
reconstructed from the ITCH feed.  Take the **AMZN** and **INTC** pairs at level 10 for the
whole session and put them in `data/lobster/`.  `data/` is git-ignored, so the files are yours
rather than the repository's, and the cells below say what they cannot do without them.

Two instruments and not one.  A one-cent tick is a coarse grid on a \$27 stock and a fine one
on a \$223 stock, so INTC's book piles onto a few prices and AMZN's spreads over hundreds of
them.  *Large-tick* and *small-tick* are the names for that, and the difference is the subject
of §3 rather than a detail of the two files.

A LOBSTER orderbook file is $4L$ columns — ask price, ask size, bid price, bid size, repeated
outward from the touch — and no clock.  One row is one configuration.  The clock is in the
message file beside it, one message to a row, and the two are aligned by position: row $i$ of
the book is the state after message $i$.

So the pair is read together.  `LobsterMarketSession` holds it as the files hold it, indexed
by position, and `coarsened` returns the `MarketSession` the rest of the package uses: the
same states on the message file's clock, with the fills of one market order collapsed onto the
row they share.  Why they share one is the next notebook's subject; here it is the clock that
is wanted.

In [ ]:
DATA = Path("../..") / "data" / "lobster"
SPEC = SessionStatistics((GridDepth(1), GridDepth(5)), (), ())
TICKERS = ("AMZN", "INTC")


def message_file(ticker):
    return DATA / f"{ticker}_2012-06-21_34200000_57600000_message_10.csv"


def recorded_session(ticker):
    files = lobster.LobsterFiles.parse(message_file(ticker))
    raw = LobsterMarketSession.from_files(files, SPEC, GRID, lobster.NASDAQ_REGULAR_HOURS)
    return raw.coarsened(True)


absent = [ticker for ticker in TICKERS if not message_file(ticker).exists()]
sessions = {} if absent else {ticker: recorded_session(ticker) for ticker in TICKERS}

if absent:
    print("absent, so every figure below is skipped:", ", ".join(absent))
    print("they are the level-10 pairs of 2012-06-21 from https://lobsterdata.com")
else:
    for ticker, session in sessions.items():
        clock = session.lobster_book.index
        print(f"{ticker}: {len(clock):,} states, from {clock[0]:.0f}s to {clock[-1]:.0f}s "
              f"after midnight")

One row, turned into the book of §2.  Everything the notebook has computed so far is
available on it, and the two indexings of §3 part company immediately.

In [ ]:
# The first row of AMZN on 2012-06-21, so that this much runs with no data at all.
FIRST_ROW = (
    2239500, 100, 2231800, 100, 2239900, 100, 2230700, 200, 2240000, 220,
    2230400, 100, 2242500, 100, 2230000, 10, 2244000, 547, 2226200, 100,
    2245400, 100, 2213000, 4000, 2248900, 100, 2204000, 100, 2267700, 100,
    2202500, 5000, 2294300, 100, 2202000, 100, 2298000, 100, 2189700, 100,
)

if sessions:
    row = sessions["AMZN"].lobster_book.iloc[0]
    UNIT = sessions["AMZN"].price_unit
else:
    row = pd.Series(FIRST_ROW, index=frames.lobster_book_columns(DEPTH))
    UNIT = lobster.price_unit(GRID)

amzn = AggregateBook.from_lobster_row(row, UNIT, DEPTH)

print("price unit         :", UNIT, "file units per tick")
print("best bid / best ask:", amzn.best_bid_price, "/", amzn.best_ask_price, "ticks")
print("                   :", round(GRID.to_price(amzn.best_bid_price), 2), "/",
      round(GRID.to_price(amzn.best_ask_price), 2), "dollars")
print("spread             :", amzn.spread, "ticks")
print("mid-price          :", amzn.mid_price, "ticks")
print("I^1                :", round(amzn.queue_imbalance(GridDepth(1)), 4))
print()
print("occupied ask levels:", amzn.occupied_levels(SELL, DEPTH)[:4])
print("grid span, ask side:", amzn.grid_span(SELL, DEPTH), "positions for", DEPTH, "levels")
print("gaps, ask side     :", amzn.gap_count(SELL, DEPTH))

The spread is odd, so the mid sits on a half tick.  `to_price` takes an integer count of
ticks and would refuse it: the mid-price is a derived quantity, not a price at which anything
can rest or trade, and the grid is under no obligation to carry it.

Ten reported levels spanning several hundred grid positions.  This is a small-tick instrument,
and the distinction of §3 is not a corner case on it: $I^2$ read off the columns of this file
pairs the touch with a price some way from it, and is not the $I^2$ of the notes.

The row round-trips.

In [ ]:
assert amzn.to_lobster_row(UNIT, DEPTH) == row.tolist()
print("round trip holds over", len(row), "columns")

### The same book at three hours

A session is not homogeneous, and the open, the middle of the day and the close are three
different markets.  One configuration from each, read off the recorded clock.

In [ ]:
HOURS = {"09:35": 34500.0, "12:30": 45000.0, "15:55": 57300.0}


def at_the_hour(session):
    """The state the book was in at each hour, as an `AggregateBook`."""
    return {
        label: AggregateBook.from_lobster_row(
            session.lobster_book.iloc[session.lobster_book.index.searchsorted(seconds)],
            session.price_unit,
            DEPTH,
        )
        for label, seconds in HOURS.items()
    }


def described(books):
    return pd.DataFrame({
        label: {
            "spread (ticks)": book.spread,
            "size at the touch": book.best_bid_size + book.best_ask_size,
            "bid grid span": book.grid_span(BUY, DEPTH),
            "ask grid span": book.grid_span(SELL, DEPTH),
        }
        for label, book in books.items()
    })


if sessions:
    hourly = {ticker: at_the_hour(session) for ticker, session in sessions.items()}
    display(pd.concat({t: described(b) for t, b in hourly.items()}, axis=1))

The two rows to read are the spread and the grid span.  AMZN holds its ten levels over tens of
grid positions and INTC over exactly ten: on INTC the reported level $k$ *is* the grid level
$k$, and on AMZN it is not.  INTC's touch carries thousands of shares against AMZN's hundreds,
which is the same fact from the other side — a queue is deep when the grid is coarse, because
there is nowhere finer to stand.

In [ ]:
if sessions:
    snapshots_figure(hourly["AMZN"], depth=DEPTH, title="AMZN").show()
    snapshots_figure(hourly["INTC"], depth=DEPTH, title="INTC").show()

### The touch through the session

The four quantities of §4, drawn against the clock: the two best prices, the mid-price between
them, and the micro-price, which sits inside the spread on the side that is thin.

A session is a quarter of a million states, which is more points than a line can carry, so the
whole day is drawn on a clock of its own: the last state of each ten-second bucket.  That the
rows can be chosen this way at all is a property of the frame — a session figure takes row
positions, and a slice is only the commonest way to name them.

In [ ]:
def on_a_clock(session, step):
    """The last row of each `step`-second bucket."""
    times = session.lobster_book.index.to_numpy()
    return np.flatnonzero(np.diff(np.floor(times / step), append=np.inf) > 0)


def between(session, opening, closing):
    """Every row whose state was the book's between two times, in seconds after midnight."""
    times = session.lobster_book.index.to_numpy()
    return np.flatnonzero((times >= opening) & (times < closing))


if sessions:
    for ticker, session in sessions.items():
        stats = session.stats
        print(f"{ticker}: the mid-price covered "
              f"{stats['MidPrice'].max() - stats['MidPrice'].min():.0f} ticks over the day, "
              f"and the spread was one tick on {(stats['Spread'] == 1).mean():.0%} of states")
        figure = touch_figure(session, on_a_clock(session, 10.0))
        figure.update_layout(title=f"{ticker}: the whole session, one point per ten seconds")
        figure.show()


Five hundred ticks against a hundred, on the same grid on the same day: the tick is a unit of
price and not a unit of movement, and it measures something different in the two books.  On
INTC the four lines are hard to tell apart at this scale.  A spread of one tick leaves the
mid-price with two values to take and the micro-price with the whole interval between them,
which is the case the micro-price is for, and it is why the next figure is two minutes rather
than six and a half hours.

In [ ]:
NOON = (45000.0, 45120.0)

if sessions:
    for ticker, session in sessions.items():
        figure = touch_figure(session, between(session, *NOON))
        figure.update_layout(title=f"{ticker}: two minutes from 12:30")
        figure.show()

The step lines are the point: the book holds a state until the next message, so a sloped
segment would draw a configuration it never had.  The mid steps by half a tick when one quote
moves, and the micro-price steps on every message, because the sizes at the touch change when
the prices do not.

### The queues behind the touch

Thirty seconds of INTC, one reported level at a time: its size above, its price below, both
sides on each.  INTC and not AMZN because on INTC the reported level *is* the grid level, so
walking the dropdown outward walks outward in price as well as in occupancy.

Alongside it, how often each level's size changed at all.  The touch is where execution
happens and where a lost queue position is expensive, and what stands behind it is waiting.

In [ ]:
INTC_HALF_MINUTE = (45000.0, 45030.0)

if sessions:
    intc = sessions["INTC"]
    window = between(intc, *INTC_HALF_MINUTE)
    changed = {
        level: (
            intc.lobster_book[[f"BidSize{level}", f"AskSize{level}"]]
            .iloc[window].diff().ne(0).any(axis=1).mean()
        )
        for level in range(1, DEPTH + 1)
    }
    print("share of states at which the level's size moved, by reported level:")
    print("  " + "  ".join(f"{level}: {share:.0%}" for level, share in changed.items()))
    level_evolution_figure(intc, window).show()

### $I^n$, and §3 measured

The imbalance of §4 at $n = 5$, drawn twice: once over the first five positions of the price
grid, which is what the notes define, and once over the first five size *columns* of the file,
which is what a column slice gives.  Both stay in $[-1, 1]$ and both move with the market, so
nothing in the output says which is which.

In [ ]:
if sessions:
    for ticker, session in sessions.items():
        grid = session.stats["QueueImbalance5"]
        columns = session.column_sliced_imbalance(GridDepth(5))
        holes = session.stats[["BidGapCount", "AskGapCount"]].sum(axis=1) > 0
        print(f"{ticker}: the reported levels are not contiguous on {holes.mean():>6.1%} of "
              f"states, and the two readings of I^5 differ on "
              f"{(grid - columns).abs().gt(1e-9).mean():>6.1%}")
        imbalance_figure(session, GridDepth(5), between(session, *NOON)).show()

On INTC the two lines are one line.  On AMZN they are two, and neither is wrong: they are
different statistics, and only one of them is the $I^n$ that section 1.4 of the notes is
written about.  A study that reports the second under the first's name has made an error that
nothing will catch.

## 9. The padding sentinels

The edge case, now that the ordinary day has been seen.  Where a side has fewer than $L$
occupied levels, LOBSTER pads.  The two sentinels have **opposite signs**, so the obvious
filter — keep the positive prices — removes one of them and keeps the other.  `frames` states
the layout once, so we do not restate the stride by hand.


In [ ]:
print("padding row at depth 2:", frames.lobster_padding_row(ReportedDepth(2)))

padded = row.copy()
padded[-4:] = frames.lobster_padding_row(ReportedDepth(1))

prices = padded[[name for name in padded.index if "Price" in name]]
sentinels = prices[prices.abs() == frames.ASK_PADDING]
print("\nsentinels present:", list(sentinels.index))
print("kept by `> 0`    :", list(sentinels[sentinels > 0].index))

`AskPrice10` survives a filter written to remove padding, and it is worth $10^{10}$.  A
statistic computed over such a frame comes out in the billions, and nothing raises.

In [ ]:
spread_in_units = padded["AskPrice1"] - padded["BidPrice1"]
bad_spread = padded["AskPrice10"] - padded["BidPrice10"]
print(f"spread at the touch : {spread_in_units:>14,} file units")
print(f"spread at level 10  : {bad_spread:>14,} file units")

In [ ]:
thin = AggregateBook.from_lobster_row(padded, UNIT, DEPTH)
print("occupied ask levels:", len(thin.occupied_levels(SELL, DEPTH)))
print("occupied bid levels:", len(thin.occupied_levels(BUY, DEPTH)))
print("spread still       :", thin.spread, "ticks")

Nine levels on each side rather than ten, and the spread unaffected: the sentinel carries size
zero, and a level of size zero is removed rather than stored.  That is the invariant doing the
work, not a special case for padding.

Whether a given file shows padding at all is a property of the instrument and the day.

In [ ]:
if sessions:
    for ticker, session in sessions.items():
        rows = (session.lobster_book.abs() == frames.ASK_PADDING).any(axis=1).sum()
        print(f"{ticker}: {rows:,} of {len(session.lobster_book):,} rows carry a sentinel")
else:
    print("the census needs the sample files")


Neither instrument, at this depth, on this day.  The format permits padding; these files do
not use it, and a pipeline that has only ever seen them has never exercised the branch that
handles it.

---

**What to take away.**  The aggregated state is closed under the arrival of an order, so it
reproduces the whole public book.  Every limit order is a market part followed by a resting
part, which is one code path and not two.  A level index is a position on the grid, and a feed
reports occupied prices — a distinction that costs nothing on INTC and changes the number on
AMZN.  And two questions this state cannot answer — whose size it is, and how much traded —
are the two that the rest of the chapter has to work around.

Next: where the sequence of configurations comes from, and why a participant computes it
rather than receiving it.
